In [1]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer.primitives import EstimatorV2 as Estimator
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.connectors import TorchConnector


def build_backbone_qnn(n_features, reps=1, entanglement="ring"):
    n_qubits = n_features
    x_params = ParameterVector("x", n_features)
    theta_params = ParameterVector("θ", length=2 * n_qubits * reps)

    qc = QuantumCircuit(n_qubits)

    # Feature encoding (same idea as before; keep simple)
    for i in range(n_qubits):
        qc.ry(x_params[i], i)

    t = 0
    for _ in range(reps):
        for q in range(n_qubits):
            qc.ry(theta_params[t], q); t += 1
            qc.rz(theta_params[t], q); t += 1

        if entanglement == "ring":
            for q in range(n_qubits - 1):
                qc.cx(q, q + 1)
            qc.cx(n_qubits - 1, 0)
        elif entanglement == "all":
            for i in range(n_qubits):
                for j in range(i + 1, n_qubits):
                    qc.cx(i, j)
        else:
            raise ValueError("entanglement must be 'ring' or 'all'")

    # IMPORTANT: use single observable (works in your environment)
    observable = SparsePauliOp.from_list([("Z" + "I"*(n_qubits-1), 1.0)])
    estimator = Estimator()

    qnn = EstimatorQNN(
        circuit=qc,
        estimator=estimator,
        observables=observable,
        input_params=list(x_params),
        weight_params=list(theta_params),
    )
    return qnn

In [2]:
class BinaryFireModel(nn.Module):
    """
    Outputs a logit (unbounded real). Use sigmoid(logit) for probability.
    """
    def __init__(self, qnn):
        super().__init__()
        self.q = TorchConnector(qnn)      # output ~[-1,1]
        self.head = nn.Linear(1, 1)       # logit = a*q + b

    def forward(self, X):
        return self.head(self.q(X))       # logits, shape (batch, 1)


def train_binary(model, X_train_a, y_train_raw, batch_size=256, epochs=10, lr=0.01):
    X = torch.tensor(X_train_a, dtype=torch.float32)
    y_bin = (np.asarray(y_train_raw) > 0).astype(np.float32)
    y = torch.tensor(y_bin, dtype=torch.float32).view(-1, 1)

    # pos_weight = (#neg / #pos) helps with imbalance
    n_pos = float(y_bin.sum())
    n_neg = float(len(y_bin) - n_pos)
    pos_weight = torch.tensor([n_neg / max(n_pos, 1.0)], dtype=torch.float32)

    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    loader = DataLoader(TensorDataset(X, y), batch_size=batch_size, shuffle=True)

    model.train()
    for epoch in range(epochs):
        total = 0.0
        n = 0
        for xb, yb in loader:
            opt.zero_grad()
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            opt.step()
            total += loss.item() * xb.shape[0]
            n += xb.shape[0]
        print(f"[binary] epoch={epoch+1:03d} loss={total/n:.6f}  pos_weight={pos_weight.item():.2f}")

    return model

In [3]:
class PositiveCountModel(nn.Module):
    """
    Predicts mean count conditional on y>0, enforced positive via softplus.
    """
    def __init__(self, qnn):
        super().__init__()
        self.q = TorchConnector(qnn)      # output ~[-1,1]
        self.head = nn.Linear(1, 1)
        self.softplus = nn.Softplus()

    def forward(self, X):
        return self.softplus(self.head(self.q(X)))  # mu(x) >= 0


def train_positive_regressor(model, X_train_a, y_train_raw, batch_size=256, epochs=10, lr=0.01):
    y = np.asarray(y_train_raw).astype(np.float32)
    pos_idx = np.where(y > 0)[0]

    if len(pos_idx) < 5:
        raise ValueError(f"Too few positive samples to train regressor: {len(pos_idx)}")

    Xpos = torch.tensor(X_train_a[pos_idx], dtype=torch.float32)
    ypos = torch.tensor(y[pos_idx], dtype=torch.float32).view(-1, 1)

    loss_fn = nn.PoissonNLLLoss(log_input=False, full=True)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    loader = DataLoader(TensorDataset(Xpos, ypos), batch_size=min(batch_size, len(pos_idx)), shuffle=True)

    model.train()
    for epoch in range(epochs):
        total = 0.0
        n = 0
        for xb, yb in loader:
            opt.zero_grad()
            mu = model(xb)
            loss = loss_fn(mu, yb)
            loss.backward()
            opt.step()
            total += loss.item() * xb.shape[0]
            n += xb.shape[0]
        print(f"[pos-reg] epoch={epoch+1:03d} loss={total/n:.6f}  n_pos={len(pos_idx)}")

    return model

In [4]:
def predict_expected_fires(binary_model, pos_model, X_a, batch_size=512):
    """
    y_hat = sigmoid(logit(x)) * mu_pos(x)
    """
    binary_model.eval()
    pos_model.eval()

    X = torch.tensor(X_a, dtype=torch.float32)
    n = X.shape[0]

    out = []
    with torch.no_grad():
        for start in range(0, n, batch_size):
            xb = X[start:start + batch_size]
            logits = binary_model(xb)
            p = torch.sigmoid(logits)          # (batch,1)
            mu = pos_model(xb)                 # (batch,1)
            yhat = (p * mu).cpu().numpy()
            out.append(yhat)

    return np.vstack(out).ravel()

In [5]:
from sklearn.metrics import mean_squared_error, r2_score, roc_auc_score

def evaluate_two_stage(binary_model, pos_model, X_test_a, y_test_raw, batch_size=512):
    y_test_raw = np.asarray(y_test_raw).astype(float)

    # expected fires prediction
    y_pred = predict_expected_fires(binary_model, pos_model, X_test_a, batch_size=batch_size)

    mse = mean_squared_error(y_test_raw, y_pred)
    r2 = r2_score(y_test_raw, y_pred)

    # AUC: use binary model score (probability) as the ranking score
    y_true_bin = (y_test_raw > 0).astype(int)

    # probability score
    binary_model.eval()
    X = torch.tensor(X_test_a, dtype=torch.float32)
    probs = []
    with torch.no_grad():
        for start in range(0, len(X), batch_size):
            xb = X[start:start+batch_size]
            probs.append(torch.sigmoid(binary_model(xb)).cpu().numpy())
    p_score = np.vstack(probs).ravel()

    auc = roc_auc_score(y_true_bin, p_score) if len(np.unique(y_true_bin)) == 2 else np.nan

    return {"mse": mse, "r2": r2, "auc_roc": auc}, y_pred, p_score

In [6]:
import pandas as pd

n_samples = 2000  # subset — full 155K rows is impractical for QNN

# Shuffle before sampling so we get a mix of fire/no-fire rows
df_full = pd.read_csv("features.csv").sample(n=n_samples, random_state=42).reset_index(drop=True)
df_x    = df_full[["month_sin","month_cos","year","avg_tmax_c","temp_range",
                    "tot_prcp_mm","dryness_3m_avg","latitude","longitude"]]

x = df_x.values.astype(float)
y = df_full["target"].values.astype(float)

# split and scale
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.1, random_state=0)

x_scaler = StandardScaler()
x_train_s = x_scaler.fit_transform(x_train)
x_test_s  = x_scaler.transform(x_test)

angle_scaler = MinMaxScaler(feature_range=(0, 2 * np.pi))
x_train_a = angle_scaler.fit_transform(x_train_s)
x_test_a  = angle_scaler.transform(x_test_s)

In [7]:
n_features = x_train_a.shape[1]  # you said 9
reps = 1

# Build separate QNNs (simplest; avoids entangling the objectives)
qnn_bin = build_backbone_qnn(n_features=n_features, reps=reps, entanglement="ring")
qnn_pos = build_backbone_qnn(n_features=n_features, reps=reps, entanglement="ring")

binary_model = BinaryFireModel(qnn_bin)
pos_model = PositiveCountModel(qnn_pos)

# Train
binary_model = train_binary(binary_model, x_train_a, y_train, batch_size=256, epochs=10, lr=0.01)
pos_model = train_positive_regressor(pos_model, x_train_a, y_train, batch_size=64, epochs=5, lr=0.01)

# Evaluate
metrics, y_pred, p_score = evaluate_two_stage(binary_model, pos_model, x_test_a, y_test, batch_size=512)
print(metrics)

No gradient function provided, creating a gradient function. If your Estimator requires transpilation, please provide a pass manager.
No gradient function provided, creating a gradient function. If your Estimator requires transpilation, please provide a pass manager.


[binary] epoch=001 loss=1.444236  pos_weight=89.00
[binary] epoch=002 loss=1.432765  pos_weight=89.00
[binary] epoch=003 loss=1.421438  pos_weight=89.00
[binary] epoch=004 loss=1.416300  pos_weight=89.00
[binary] epoch=005 loss=1.408221  pos_weight=89.00
[binary] epoch=006 loss=1.396664  pos_weight=89.00
[binary] epoch=007 loss=1.393487  pos_weight=89.00
[binary] epoch=008 loss=1.388146  pos_weight=89.00
[binary] epoch=009 loss=1.383733  pos_weight=89.00
[binary] epoch=010 loss=1.377463  pos_weight=89.00
[pos-reg] epoch=001 loss=1.055824  n_pos=20
[pos-reg] epoch=002 loss=1.053697  n_pos=20
[pos-reg] epoch=003 loss=1.051528  n_pos=20
[pos-reg] epoch=004 loss=1.049464  n_pos=20
[pos-reg] epoch=005 loss=1.047400  n_pos=20
{'mse': 0.053518774481548875, 'r2': -21.39041549896087, 'auc_roc': 0.4120603015075377}


In [13]:
import numpy as np
import torch

def predict_expected_fires(model, X_all_a, batch_size=1024):
    """
    Returns expected-fire-count predictions for every row in X_all_a.

    model should output raw fire counts (e.g., includes Linear + Softplus head).
    """
    model.eval()

    X = torch.tensor(X_all_a, dtype=torch.float32)
    n = X.shape[0]

    preds = []
    with torch.no_grad():
        for start in range(0, n, batch_size):
            xb = X[start:start + batch_size]
            yb = model(xb)  # (batch, 1)
            preds.append(yb.detach().cpu().numpy())

    y_pred_all = np.vstack(preds).ravel()
    return y_pred_all

df = pd.read_csv("features.csv")
dfx = df[["month_sin","month_cos","year","avg_tmax_c","temp_range",
                    "tot_prcp_mm","dryness_3m_avg","latitude","longitude"]]
x_a = dfx.values.astype(float)
#y_a = df_full["target"].values.astype(float)

y_expected_all = predict_expected_fires(binary_model, x_a, batch_size=1024)


print(y_expected_all.shape)
print("first 10 predictions:", y_expected_all[:1000])
print(max(y_expected_all))

(155580,)
first 10 predictions: [-1.52479842e-01 -1.43578514e-01 -1.06258199e-01 -6.69121891e-02
 -4.55139577e-02  1.30100846e-02  2.89461017e-03 -2.02766061e-02
 -3.48311961e-02 -5.20445108e-02 -8.81889910e-02 -1.29179016e-01
 -1.20779797e-01 -9.27342176e-02 -9.40802544e-02 -8.46442729e-02
 -1.26005411e-02  4.30758297e-02  7.80393183e-02  5.62355518e-02
 -1.85006857e-03 -1.00827411e-01 -9.74810570e-02 -1.18742138e-01
 -1.37805119e-01 -1.52490333e-01 -1.14733696e-01 -6.47029579e-02
  1.44517422e-02  4.34765518e-02  5.45346141e-02  8.67196918e-03
 -3.70234251e-04 -5.10682464e-02 -1.03733912e-01 -1.27408624e-01
 -1.62497625e-01 -1.49353534e-01 -1.26325026e-01 -9.74997878e-02
 -3.01296115e-02 -2.91761756e-03  4.38053310e-02  4.20308113e-02
  1.13700032e-02 -5.18432260e-02 -1.45778522e-01 -1.22820690e-01
 -1.51815027e-01 -1.83591679e-01 -1.44742757e-01 -9.03889686e-02
 -6.19142354e-02 -3.39224041e-02 -1.22966468e-02 -3.92762423e-02
 -5.33930063e-02 -8.65303427e-02 -1.22271076e-01 -1.630852

In [12]:
# prints ALL trainable parameters (names + full tensors)

for name, p in binary_model.named_parameters():
    if p.requires_grad:
        print(name)
        print(p.detach().cpu())
        print()

for name, p in pos_model.named_parameters():
    if p.requires_grad:
        print(name)
        print(p.detach().cpu())
        print()

q.weight
tensor([-0.0278,  0.5805, -0.0250, -0.8745, -0.3843, -0.8448,  0.1502,  0.1234,
        -0.0090, -0.8322, -0.0837,  0.8424, -0.1121,  0.6449,  0.5712, -0.8542,
        -0.8517, -0.1751])

head.weight
tensor([[0.8032]])

head.bias
tensor([-0.3156])

q.weight
tensor([-0.8661, -0.7050, -0.8626, -0.2675,  0.3043,  0.8662, -0.1759, -0.6120,
         0.5089,  0.1406,  0.8944,  0.1867, -0.8047,  0.1662,  0.8076, -0.7162,
         0.4557, -0.4831])

head.weight
tensor([[-0.0037]])

head.bias
tensor([-0.3272])



In [41]:
import numpy as np
import pandas as pd
import torch

def export_location_timeseries_csv_two_stage(
    binary_model,
    pos_model,
    df,
    x_scaler,
    angle_scaler,
    lat_value,
    lon_value,
    tol=1e-6,
    year_range=None,         # optional (min_year, max_year)
    row_range=None,          # optional slice AFTER sorting
    batch_size=1024,
    out_csv_path="lat_lon_timeseries.csv",
):
    # exact columns from your CSV
    feature_cols = [
        "month_sin","month_cos","year","avg_tmax_c","temp_range",
        "tot_prcp_mm","dryness_3m_avg","latitude","longitude"
    ]
    date_col = "year_month"

    # sanity
    missing = [c for c in feature_cols + [date_col] if c not in df.columns]
    if missing:
        raise ValueError(f"df missing columns: {missing}")

    X_all = df[feature_cols].to_numpy(dtype=float)         # (N,9)
    ym_all = df[date_col].astype(str).to_numpy()           # (N,)

    # filter by lat/lon (feature indices 7 and 8)
    lat = X_all[:, 7]
    lon = X_all[:, 8]
    mask = (np.abs(lat - lat_value) <= tol) & (np.abs(lon - lon_value) <= tol)

    idx = np.where(mask)[0]
    if len(idx) == 0:
        raise ValueError("No rows matched the given (lat, lon).")

    Xf = X_all[idx]
    ym = ym_all[idx]

    # parse YYYY-MM
    years = np.array([int(s[:4]) for s in ym], dtype=int)
    months = np.array([int(s[5:7]) for s in ym], dtype=int)

    if year_range is not None:
        y0, y1 = year_range
        keep = (years >= y0) & (years <= y1)
        Xf = Xf[keep]
        years = years[keep]
        months = months[keep]
        if len(Xf) == 0:
            raise ValueError("year_range produced an empty selection.")

    # sort by time
    order = np.lexsort((months, years))
    Xf = Xf[order]
    years = years[order]
    months = months[order]

    if row_range is not None:
        a, b = row_range
        Xf = Xf[a:b]
        years = years[a:b]
        months = months[a:b]
        if len(Xf) == 0:
            raise ValueError("row_range produced an empty selection.")

    # month index from start (no rollover)
    month_from_start = ((years - years[0]) * 12 + (months - months[0])).astype(int)

    if len(np.unique(month_from_start)) != len(month_from_start):
        raise ValueError("Duplicate month_from_start detected (duplicate year_month rows).")

    # scale and predict
    Xa = angle_scaler.transform(x_scaler.transform(Xf))

    binary_model.eval()
    pos_model.eval()
    Xt = torch.tensor(Xa, dtype=torch.float32)

    preds = []
    with torch.no_grad():
        for start in range(0, len(Xt), batch_size):
            xb = Xt[start:start + batch_size]
            p = torch.sigmoid(binary_model(xb))
            mu = pos_model(xb)
            preds.append((p * mu).cpu().numpy().ravel())
    y_pred = np.concatenate(preds)

    out = pd.DataFrame({"month_from_start": month_from_start, "prediction": y_pred.astype(float)})
    out.to_csv(out_csv_path, index=False)
    return out

df_raw = pd.read_csv("features.csv")

out_df = export_location_timeseries_csv_two_stage(
    binary_model=binary_model,
    pos_model=pos_model,
    df=df_raw,
    x_scaler=x_scaler,
    angle_scaler=angle_scaler,
    lat_value=0.15040841135447547,
    lon_value=0.5939119939852004,
    tol=1e-6,
    row_range=(0, 300),
    out_csv_path="lat_lon_timeseries.csv",
)

In [40]:
print(df.columns.tolist())
print(df.head(2))

['month_from_start', 'prediction']
   month_from_start  prediction
0                 0    0.229216
1                 0    0.230133
